In [ ]:
import json
from datasets import Dataset
from transformers import BertTokenizerFast, BertForQuestionAnswering, Trainer, TrainingArguments
import torch

import transformers
print(transformers.__version__)

# 1. JSON 파일 로드
with open("update_qa_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. 데이터셋 변환
qa_examples = []
for item in data:
    qa_examples.append({
        "id": item["id"],
        "context": item["context"],
        "question": item["question"],
        "answers": {
            "text": [ans["text"] for ans in item["answers"]],
            "answer_start": [ans["answer_start"] for ans in item["answers"]],
        }
    })

dataset = Dataset.from_list(qa_examples)

# 3. 토크나이저 로드
model_name = "beomi/kcbert-base"
tokenizer = BertTokenizerFast.from_pretrained(model_name)

# 4. 데이터 전처리 함수
def preprocess_function(examples):
    questions = [q.lstrip() for q in examples["question"]]
    contexts = examples["context"]

    max_length = 300  # 수정: kcbert는 300까지만 지원
    doc_stride = 128

    tokenized_examples = tokenizer(
        questions,
        contexts,
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        answer_starts = answers["answer_start"]
        answer_texts = answers["text"]

        if len(answer_starts) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            start_char = answer_starts[0]
            answer_text = answer_texts[0]
            end_char = start_char + len(answer_text)

            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                start_positions.append(cls_index)
                end_positions.append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                start_positions.append(token_start_index - 1)

                while token_end_index >= 0 and offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                end_positions.append(token_end_index + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions
    return tokenized_examples

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

# 5. 모델 로드
model = BertForQuestionAnswering.from_pretrained(model_name)

# 6. Trainer 설정
training_args = TrainingArguments(
    output_dir="./qa_kcbert_finetuned",
    eval_strategy="no",  # 또는 "epoch" (평가를 원할 경우)
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    save_steps=500,
    logging_steps=100,
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

# 7. 학습 실행
trainer.train()

# 8. 저장
trainer.save_model("./qa_kcbert_finetuned")
tokenizer.save_pretrained("./qa_kcbert_finetuned")

In [7]:
import transformers
print(transformers.__version__)  # 실제 문자열 출력 확인
print(transformers.TrainingArguments.__init__.__code__.co_varnames)  # init 함수 파라미터 목록 확인

4.53.3
('self', 'output_dir', 'overwrite_output_dir', 'do_train', 'do_eval', 'do_predict', 'eval_strategy', 'prediction_loss_only', 'per_device_train_batch_size', 'per_device_eval_batch_size', 'per_gpu_train_batch_size', 'per_gpu_eval_batch_size', 'gradient_accumulation_steps', 'eval_accumulation_steps', 'eval_delay', 'torch_empty_cache_steps', 'learning_rate', 'weight_decay', 'adam_beta1', 'adam_beta2', 'adam_epsilon', 'max_grad_norm', 'num_train_epochs', 'max_steps', 'lr_scheduler_type', 'lr_scheduler_kwargs', 'warmup_ratio', 'warmup_steps', 'log_level', 'log_level_replica', 'log_on_each_node', 'logging_dir', 'logging_strategy', 'logging_first_step', 'logging_steps', 'logging_nan_inf_filter', 'save_strategy', 'save_steps', 'save_total_limit', 'save_safetensors', 'save_on_each_node', 'save_only_model', 'restore_callback_states_from_checkpoint', 'no_cuda', 'use_cpu', 'use_mps_device', 'seed', 'data_seed', 'jit_mode_eval', 'use_ipex', 'bf16', 'fp16', 'fp16_opt_level', 'half_precision_ba

In [ ]:
import json

with open("update_qa_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

new_data = []
for item in data:
    # 답변 여러 개 중 첫 번째 답변만 사용
    first_answer = item["answers"][0] if item["answers"] else {"text": "", "answer_start": -1}
    
    new_item = {
        "context": item["context"],
        "question": item["question"],
        "answers": {
            "text": [first_answer["text"]],
            "answer_start": [first_answer["answer_start"]]
        }
    }
    new_data.append(new_item)

with open("light_qa_data.json", "w", encoding="utf-8") as f_out:
    json.dump(new_data, f_out, ensure_ascii=False, indent=2)